# SO-101 trên Google Colab: ghi demo, train ACT / SmolVLA, DAgger, chấm, đẩy lên Hugging Face

Chạy ô 1-3 mỗi lần mở (hoặc sau khi Colab ngắt), rồi chạy mục cần làm. Mỗi mục chạy **nền** (đóng tab vẫn chạy), lưu mọi thứ lên
Drive `MyDrive/vla/` theo từng phần, nên bị ngắt thì chạy lại ô 1-3 và đúng ô đó: phần đã xong được bỏ qua, train chạy tiếp từ checkpoint.

| Mục | Làm gì | Thời gian (L4) |
|---|---|---|
| 4 | Ghi demo bằng policy viết tay (thầy giáo) | 500 ván ~12 phút |
| 5 | Train ACT từ đầu + chấm mọi checkpoint | 60k step ~4 giờ + chấm |
| 6 | Một vòng DAgger: AI lái, thầy sửa, train tiếp, chấm | ~4-5 giờ |
| 7 | Chấm một mạng / gộp nhiều checkpoint trên 774 seed | ~40 phút một mạng |
| 8 | Fine-tune SmolVLA + chấm | chưa đo |
| 9 | Đẩy dataset lên Hugging Face | vài phút |
| 10 | Xem tiến độ mọi việc đang chạy | |

Kết quả tới giờ và lý do từng lựa chọn: `docs/so_tay_ai.md`. Tốt nhất: ACT gộp 3 checkpoint DAgger, **84,8 %** trên 774 seed.

**Trước khi chạy:** *Runtime -> Change runtime type -> GPU* (**L4** hoặc A100).

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Nối Google Drive

Dataset, checkpoint, kết quả chấm đều nằm trong `MyDrive/vla` (giống `D:/vla` ở máy nhà). Colab hỏi quyền Drive: bấm cho phép.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
VLA = '/content/drive/MyDrive/vla'          # thư mục gốc trên Drive
for sub in ('datasets', 'outputs/train', 'eval', 'replays'):
    os.makedirs(f'{VLA}/{sub}', exist_ok=True)
print('Drive sẵn sàng:', VLA)

## 3. Lấy code, cài thư viện, hàm dùng chung

Code lấy từ GitHub (nhánh `my-work`), thư mục phải tên `sim_engine` (tên gói Python). `MUJOCO_GL=egl`: Colab không có màn hình,
MuJoCo vẽ ảnh camera bằng GPU. Cài ~3-5 phút; Colab đòi *Restart session* thì restart rồi chạy lại từ ô 2.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['HF_HOME'] = '/content/hf-cache'
# Khai báo driver EGL của NVIDIA: thiếu file này MuJoCo vẽ ảnh bằng CPU, chậm ~160 lần
p = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(p):
    open(p, 'w').write('{"file_format_version" : "1.0.0", "ICD" : {"library_path" : "libEGL_nvidia.so.0"}}')
if os.path.isdir('/content/sim_engine/.git'):
    !git -C /content/sim_engine pull -q
else:
    !git clone -q -b my-work https://github.com/tvhhai/sim-engine.git /content/sim_engine
!git -C /content/sim_engine log --oneline -1
# smolvla: thư viện phụ cho mục 8 (transformers, accelerate); ACT không cần nhưng cài chung cho gọn
!pip install -q "lerobot[dataset,training,smolvla]==0.7.0" "mujoco>=4.0.1" gymnasium opencv-python-headless h5py

In [ ]:
# Hàm dùng chung: mỗi mục ghép các dòng lệnh shell rồi chạy nền bằng bg().
import subprocess
D = '/content/datasets'                      # dataset giải nén ra ổ của máy Colab (đọc nhanh hơn từ Drive)
T = f'{VLA}/outputs/train'                   # checkpoint lưu thẳng lên Drive

def bg(name, lines):
    """Ghi các dòng lệnh vào /content/<name>.sh và chạy nền; log ở /content/<name>.log (mục 10 đọc)."""
    head = ['set -x', 'export MUJOCO_GL=egl HF_HOME=/content/hf-cache', f'cd /content && mkdir -p {D}']
    open(f'/content/{name}.sh', 'w').write('\n'.join(head + lines + ['echo ALL_DONE']) + '\n')
    subprocess.Popen(f'bash /content/{name}.sh > /content/{name}.log 2>&1', shell=True, start_new_session=True)
    print('Đã chạy nền:', name, '| xem tiến độ ở mục 10')

def unzip(pattern):
    """Giải nén mọi zip khớp pattern (trên Drive) vào D, cái nào đã giải nén thì bỏ qua."""
    return (f'for Z in {VLA}/datasets/{pattern}.zip; do P={D}/$(basename $Z .zip); '
            f'[ -d $P ] || (mkdir -p $P && unzip -q $Z -d $P) || exit 1; done')

def merge(name, sources):
    """Gộp nhiều dataset (đường dẫn / glob trong D) thành D/name, đã có thì bỏ qua."""
    return (f'[ -d {D}/{name} ] || python -m sim_engine.learning.merge_datasets --out {D}/{name} --repo-id local/{name} '
            + ' '.join(f'{D}/{s}' for s in sources) + ' || exit 1')

def train(cfg, out, ds, extra=''):
    """Train theo config YAML; out đã có checkpoint thì train tiếp (resume)."""
    tc = f'{out}/checkpoints/last/pretrained_model/train_config.json'
    return [f'if [ -f {tc} ]; then lerobot-train --config_path={tc} --resume=true --dataset.root={D}/{ds} || exit 1; else',
            f'  rm -rf {out}; python -m sim_engine.learning.train sim_engine/learning/configs/{cfg}.yaml --dataset.repo_id=local/{ds} '
            f'--dataset.root={D}/{ds} --output_dir={out} --job_name=$(basename {out}) {extra} || exit 1; fi']

def score(out, tag, par=3):
    """Chấm mọi checkpoint của out trên 774 seed (par lượt song song) -> eval/full_<tag>_<step>.json; đã chấm thì bỏ qua."""
    return [f'ls {out}/checkpoints | grep -E "^[0-9]+$" | while read k; do [ -f {VLA}/eval/full_{tag}_$k.json ] || echo $k; done '
            f'| xargs -P {par} -I KK sh -c "python -m sim_engine.learning.eval_policy --policy {out}/checkpoints/KK/pretrained_model '
            f'--name {tag}_KK --out {VLA}/eval/full_{tag}_KK.json > /content/full_{tag}_KK.log 2>&1"',
            f'for f in {VLA}/eval/full_{tag}_*.json; do echo "CK $(basename $f) $(grep -m1 \'"all"\' $f)"; done']

# Kiểm tra nhanh: dựng env, vẽ một ảnh camera
%cd /content
from sim_engine.learning import sim_setup
env, opt = sim_setup.make_env(3000)
obs, _ = env.reset(seed=3000, options=opt)
print({k: getattr(v, 'shape', v) for k, v in sim_setup.to_frame(env, obs, [0] * 7, 'test', ('top', 'wrist')).items()})
env.close()

## 4. Ghi demo (thầy giáo = policy viết tay mặc định, bản min-jerk)

`learning/record_dataset.py`, chỉ giữ ván thành công. Chia `PARTS` phần chạy song song, mỗi phần một dải seed riêng (3000+; bộ chấm
dùng 0-2499, DAgger dùng 6000+), xong phần nào zip lên Drive phần đó, cuối cùng gộp thành `D/NAME`.
`so101_box4_topwrist_mj` (5 x 100 ván) đã có trên Drive: chạy ô này chỉ giải nén và gộp.

In [ ]:
NAME, PARTS, PER_PART = 'so101_box4_topwrist_mj', 5, 100
part = [f'rec() {{ P={D}/{NAME}_p$1; Z={VLA}/datasets/{NAME}_p$1.zip',
        '  if [ -f $Z ]; then [ -d $P ] || (mkdir -p $P && unzip -q $Z -d $P); return; fi',
        f'  rm -rf $P; python -m sim_engine.learning.record_dataset --episodes {PER_PART} --cameras top,wrist '
        f'--seed-start $((3000 + $1 * 400)) --root $P --repo-id local/{NAME}_p$1 > /content/rec_p$1.log 2>&1 || return 1',
        '  (cd $P && zip -qr /content/rec_p$1.zip . && cp /content/rec_p$1.zip $Z.tmp && mv $Z.tmp $Z); echo "Xong ghi phần $1"; }']
bg('record', part + [f'for K in $(seq 0 {PARTS - 1}); do rec $K & done; wait',
                     f'for K in $(seq 0 {PARTS - 1}); do [ -f {VLA}/datasets/{NAME}_p$K.zip ] || exit 1; done',
                     merge(NAME, [f'{NAME}_p[0-9]*'])])

## 5. Train ACT từ đầu + chấm mọi checkpoint

`learning/configs/act_scratch.yaml` (camera trên cao + cổ tay, 60k step). Chấm mọi checkpoint trên 774 seed: điểm giữa các checkpoint
chênh nhau nhiều, checkpoint cuối không phải tốt nhất. Lưu ý (sổ tay mục 13): train từ đầu chỉ ~59 %, phải làm tiếp DAgger (mục 6).

In [ ]:
DS, OUT, TAG = 'so101_box4_topwrist_mj', f'{T}/act_scratch_mj_colab', 'scratch'
bg('scratch', [unzip(f'{DS}_p[0-9]*'), merge(DS, [f'{DS}_p[0-9]*'])] + train('act_scratch', OUT, DS) + score(OUT, TAG))

## 6. Một vòng DAgger: AI lái, thầy sửa, train tiếp, chấm

`learning/dagger.py`: mỗi ván AI tự lái một đoạn ngẫu nhiên (15-250 bước) rồi thầy tiếp quản và làm nốt, ghi đoạn thầy làm. Ghi
`PARTS` x `CHUNK` ván (3 phần song song), gộp với demo gốc + mọi vòng trước, train tiếp `BASE` bằng `act_finetune.yaml` (lr giảm dần),
chấm mọi checkpoint. Vòng sau: tăng `ROUND`, đổi `DRIVERS` / `BASE` sang mạng tốt nhất vừa có (xem kết quả ở mục 10).

In [ ]:
ROUND = 4
# AI lái khi ghi DAgger: một hoặc nhiều checkpoint (nhiều = gộp, trung bình action). Vòng 4: gộp 3 checkpoint tốt nhất (84,8 %).
DRIVERS = [f'{T}/act_box4_topwrist_500_dagger2_colab/checkpoints/010000/pretrained_model',
           f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/020000/pretrained_model',
           f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/030000/pretrained_model']
BASE = f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/020000/pretrained_model'   # điểm xuất phát để train tiếp
DEMO = 'so101_box4_topwrist_500'      # demo gốc của chuỗi DAgger này (500 ván)
PARTS, CHUNK, STEPS = 10, 50, 20000
DAG = lambda r: f'so101_box4_topwrist_dagger{r}'
MERGED, OUT = f'so101_box4_topwrist_500_dagger{ROUND}', f'{T}/act_box4_topwrist_500_dagger{ROUND}_colab'
SEED0 = 6000 + (ROUND - 1) * 10000    # mỗi vòng một dải seed riêng, mỗi phần 1000 seed

open('/content/dag_part.sh', 'w').write('\n'.join([
    f'K=$1; P={D}/{DAG(ROUND)}_p$K; Z={VLA}/datasets/{DAG(ROUND)}_p$K.zip',
    'if [ -f $Z ]; then [ -d $P ] || (mkdir -p $P && unzip -q $Z -d $P); exit 0; fi',
    f'rm -rf $P; python -m sim_engine.learning.dagger --policy {" ".join(DRIVERS)} --episodes {CHUNK} '
    f'--seed-start $(({SEED0} + K * 1000)) --root $P --repo-id local/{DAG(ROUND)}_p$K > /content/dag_p$K.log 2>&1 || exit 1',
    '(cd $P && zip -qr /content/dag_p$K.zip . && cp /content/dag_p$K.zip $Z.tmp && mv $Z.tmp $Z && rm /content/dag_p$K.zip) || exit 1',
]) + '\n')
bg(f'dagger{ROUND}', [
    f'mkdir -p {D}/{DEMO} && [ -f {D}/{DEMO}/meta/info.json ] || unzip -q {VLA}/datasets/{DEMO}.zip -d {D}/{DEMO}',
    *[unzip(f'{DAG(r)}_p[0-9]*') for r in range(1, ROUND)],
    f'seq 0 {PARTS - 1} | xargs -P 3 -I KK bash /content/dag_part.sh KK',
    f'for K in $(seq 0 {PARTS - 1}); do [ -f {VLA}/datasets/{DAG(ROUND)}_p$K.zip ] || exit 1; done',
    merge(MERGED, [DEMO] + [f'{DAG(r)}_p[0-9]*' for r in range(1, ROUND + 1)]),
    *train('act_finetune', OUT, MERGED, f'--policy.path={BASE} --steps={STEPS}'),
    *score(OUT, f'd{ROUND}'),
])

## 7. Chấm một mạng, hoặc gộp nhiều checkpoint, trên 774 seed

Nhiều thư mục trong `POLICIES` = gộp: mỗi bước mọi mạng cùng đoán, chạy trung bình action (`eval_policy.act`). Gộp 3 checkpoint
DAgger: 77,8 % -> 84,8 %. Kết quả: `MyDrive/vla/eval/full_<TAG>.json`.

In [ ]:
TAG = 'ens3'
POLICIES = [f'{T}/act_box4_topwrist_500_dagger2_colab/checkpoints/010000/pretrained_model',
            f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/020000/pretrained_model',
            f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/030000/pretrained_model']
bg(f'eval_{TAG}', [f'python -m sim_engine.learning.eval_policy --policy {" ".join(POLICIES)} --name {TAG} '
                   f'--out {VLA}/eval/full_{TAG}.json > /content/full_{TAG}.log 2>&1'])

## 8. Fine-tune SmolVLA + chấm

`learning/configs/smolvla_finetune.yaml`: từ bản học sẵn `lerobot/smolvla_base` (~450M tham số), khoá phần nhìn, chỉ train phần ra
action, lr giảm dần có sẵn. Camera đổi tên top -> camera1, wrist -> camera2 cho khớp bản học sẵn. Dataset mặc định: đúng dữ liệu của
mạng ACT tốt nhất (demo gốc + DAgger 1-4) để so công bằng với 84,8 %. Chấm chậm hơn ACT (mạng to hơn).

In [ ]:
DS, OUT, TAG = 'so101_box4_topwrist_500_dagger4', f'{T}/smolvla_dagger4_colab', 'smolvla'
SOURCES = ['so101_box4_topwrist_500'] + [f'so101_box4_topwrist_dagger{r}_p[0-9]*' for r in range(1, 5)]
bg('smolvla', [f'mkdir -p {D}/so101_box4_topwrist_500 && [ -f {D}/so101_box4_topwrist_500/meta/info.json ] || '
               f'unzip -q {VLA}/datasets/so101_box4_topwrist_500.zip -d {D}/so101_box4_topwrist_500',
               *[unzip(f'so101_box4_topwrist_dagger{r}_p[0-9]*') for r in range(1, 5)],
               merge(DS, SOURCES)] + train('smolvla_finetune', OUT, DS) + score(OUT, TAG, par=2))

## 9. Đẩy dataset lên Hugging Face

Chia sẻ cho team, xem từng ván trên web, train ở máy khác bằng `--dataset.repo_id=<HF_USER>/<tên>`. Cần token **Write**: tạo ở
huggingface.co -> Settings -> Access Tokens, rồi thêm vào Colab: biểu tượng chìa khoá (Secrets) bên trái -> `HF_TOKEN`, bật Notebook
access. `PRIVATE = True`: chỉ mình xem được, đổi sang public sau ở trang Settings của repo. Đẩy 500 demo min-jerk + bản con 50 ván.

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
HF_USER, PRIVATE = 'tvhhai', True
NAME = 'so101_box4_topwrist_mj'                        # 500 demo min-jerk (mục 4)
flag = ' --private' if PRIVATE else ''
bg('push', [unzip(f'{NAME}_p[0-9]*'), merge(NAME, [f'{NAME}_p[0-9]*']),
            f'python -m sim_engine.learning.push_dataset --root {D}/{NAME} --repo-id {HF_USER}/so101_box4_mj{flag} || exit 1',
            f'python -m sim_engine.learning.push_dataset --root {D}/{NAME} --repo-id {HF_USER}/so101_box4_mj_50 --first 50{flag} || exit 1'])

## 10. Tiến độ

Mỗi việc nền có log `/content/<tên>.log` và kết thúc bằng `ALL_DONE`. Ô dưới in vài dòng cuối có ích của mọi việc: ghi (seed ...),
DAgger (từng phần), train (step: ...), chấm ([k/774] ...), điểm từng checkpoint (CK ...), lỗi.

In [ ]:
!for f in /content/*.log; do echo "== $f"; tr '\r' '\n' < $f | grep -E "^seed [0-9]+|Xong|step:|^CK |\[[0-9]+/774\]|\"all\"|Đã đẩy|ALL_DONE|Error|Traceback" | tail -n 2 | cut -c1-170; done

## 11. Đem checkpoint về máy nhà (khi cần)

Checkpoint nằm trên Drive: `MyDrive/vla/outputs/train/<tên>/checkpoints/<step>/pretrained_model/`. Tải cả thư mục `pretrained_model`
về (vd `D:/vla/outputs/train/...`) rồi chạy `eval_policy --policy <thư mục đó>` như ở nhà.